# EviChange — Person 1 (Perception & Evidence) starter notebook

**What this notebook does, in order:**

1. Connects to **Google Earth Engine** (GEE) — Google's free satellite-image cloud.
2. Gets **cloud-free Sentinel-2 images** of the Quang Ninh study area for a "before" and "after" date, without downloading huge files yourself.
3. Computes the same **spectral indices** Person 2's notebook uses (NDVI, NDBI, MNDWI).
4. Builds a **simple threshold change mask** as a first baseline (you will replace this with a trained model, e.g. U-Net, later — this gets something real into the pipeline *today*).
5. Turns the change mask into **change-region polygons with real GPS coordinates** — GeoJSON, same column names Person 2 and the web app already use.
6. Exports everything to your Google Drive, ready to send to Person 3.

**How to use:** Run cells top to bottom with **Shift+Enter**. Each cell says what you should see.

**Before you start:** you need a free Earth Engine account. If you don't have one yet: go to https://code.earthengine.google.com, sign in with a Google account, and accept the terms (pick "Academia & Research" as the use case — usually approved instantly).

## Step 1 — Connect to Earth Engine
A link will appear — click it, sign in with the Google account you registered for Earth Engine, copy the code back here. You should see `Earth Engine connected.`

In [ ]:
!pip -q install earthengine-api geemap

import ee
ee.Authenticate()               # one-time per Colab session; follow the link
ee.Initialize(project=None)     # if you have a GEE cloud project id, put it here as a string
print('Earth Engine connected.')

## Step 2 — Connect Google Drive (so results are saved permanently)
A popup will ask permission — click *Allow*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/evichange'
OUT = f'{PROJECT}/person1_outputs'
os.makedirs(OUT, exist_ok=True)
print('Folder ready:', OUT)

## Step 3 — Define the study area
This is the same box already built into the web app's demo data, so your real results will show up in the same place on the map.
**If your advisor gives you an official Quang Ninh / Cam Pha boundary (a shapefile or GeoJSON), replace this box with that** — ask Person 3 how to load a file here.

You should see a small interactive map with an orange outline.

In [ ]:
import geemap

AOI = ee.Geometry.Rectangle([107.22, 20.98, 107.40, 21.10])  # [west, south, east, north] — matches the web app's demo AOI

m = geemap.Map(center=[21.04, 107.31], zoom=11)
m.addLayer(AOI, {'color': 'orange'}, 'Study area (AOI)')
m

## Step 4 — Get cloud-free Sentinel-2 images
We ask Earth Engine for **every Sentinel-2 image over this area in each date range**, remove cloudy pixels, and take the **median** — this gives one clean, mostly cloud-free picture per period instead of one lucky (or unlucky) single scene.

Pick two periods far enough apart to show real change (this project uses roughly **2018 and 2022/2023**). A few months' range per period increases the chance of finding clear pixels.

You should see two numbers greater than 0 (`before scenes found` / `after scenes found`). If either is 0, widen the date range or the cloud filter below.

In [ ]:
BEFORE_RANGE = ('2018-01-01', '2018-12-31')
AFTER_RANGE  = ('2022-06-01', '2023-05-31')
MAX_CLOUD_PCT = 30   # scenes cloudier than this (%) are dropped before the median

S2 = 'COPERNICUS/S2_SR_HARMONIZED'   # Sentinel-2 surface reflectance, cloud-corrected

def mask_clouds(img):
    # Sentinel-2's own cloud/cirrus bit-mask band
    qa = img.select('QA60')
    cloud_bit, cirrus_bit = 1 << 10, 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    return img.updateMask(mask).divide(10000)  # scale reflectance to 0-1

def get_composite(date_range):
    col = (ee.ImageCollection(S2)
           .filterBounds(AOI)
           .filterDate(*date_range)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
           .map(mask_clouds))
    return col.median().clip(AOI), col.size()

before_img, before_n = get_composite(BEFORE_RANGE)
after_img,  after_n  = get_composite(AFTER_RANGE)
print('before scenes found:', before_n.getInfo())
print('after scenes found: ', after_n.getInfo())

You should see two true-colour pictures of the same area, at different dates, with no big cloud patches.

In [ ]:
VIS_RGB = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}

m2 = geemap.Map(center=[21.04, 107.31], zoom=11)
m2.addLayer(before_img, VIS_RGB, 'Before')
m2.addLayer(after_img, VIS_RGB, 'After')
m2.addLayer(AOI, {'color': 'orange'}, 'Study area', True, 0.4)
m2

## Step 5 — Compute spectral indices (same formulas as Person 2's notebook)
- **NDVI** — vegetation (higher = greener)
- **NDBI** — built-up / bare surface (higher = more built-up or bare soil, which is what mining exposes)
- **MNDWI** — water (higher = more water)

Then we take **after minus before** for each, exactly like `dNDVI`, `dNDBI`, `dMNDWI` in Person 2's evidence file.

In [ ]:
def indices(img):
    ndvi  = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndbi  = img.normalizedDifference(['B11', 'B8']).rename('NDBI')
    mndwi = img.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return ee.Image.cat([ndvi, ndbi, mndwi])

idx_before, idx_after = indices(before_img), indices(after_img)
d_ndvi  = idx_after.select('NDVI').subtract(idx_before.select('NDVI')).rename('dNDVI')
d_ndbi  = idx_after.select('NDBI').subtract(idx_before.select('NDBI')).rename('dNDBI')
d_mndwi = idx_after.select('MNDWI').subtract(idx_before.select('MNDWI')).rename('dMNDWI')
diff = ee.Image.cat([d_ndvi, d_ndbi, d_mndwi])

m3 = geemap.Map(center=[21.04, 107.31], zoom=11)
m3.addLayer(diff.select('dNDVI'), {'min': -0.5, 'max': 0.5, 'palette': ['red', 'white', 'green']}, 'dNDVI (red = vegetation loss)')
m3

## Step 6 — Baseline change mask (thresholds — replace with your trained model later)
A pixel counts as **changed** if any index moved more than the threshold. This is deliberately simple: it gets a first real signal into the pipeline today.
**This week / next week:** train a proper model (U-Net is the project's suggestion) on OSCD, then swap this cell for your model's prediction — everything downstream (Step 7 onward) stays the same.

`T_IDX` matches Person 2's notebook so both sides agree on what counts as "clear evidence".

In [ ]:
T_IDX = 0.10
change_mask = (d_ndvi.abs().gt(T_IDX)
               .Or(d_ndbi.abs().gt(T_IDX))
               .Or(d_mndwi.abs().gt(T_IDX))).rename('change').selfMask()

m4 = geemap.Map(center=[21.04, 107.31], zoom=11)
m4.addLayer(after_img, VIS_RGB, 'After (for context)')
m4.addLayer(change_mask, {'palette': ['red']}, 'Change mask')
m4

## Step 7 — Turn the mask into named regions (this is the file Person 3 needs)
Earth Engine groups touching changed pixels into **blobs** (connected components), turns each blob into a **polygon with real GPS coordinates**, and measures its area and average index values — directly, no manual pixel math needed.

Small specks (below `MIN_AREA_HA`) are dropped as noise. The 5 largest become `r1`…`r5`, same as Person 2's notebook.

You should see a printout of a GeoJSON `FeatureCollection` with `regions`-like features.

In [ ]:
MIN_AREA_HA = 1.0    # ignore blobs smaller than this (hectares) — raise this if you get too much speckle
MAX_REGIONS = 5
SCALE_M = 20          # Sentinel-2 working resolution in meters for this step

blobs = change_mask.connectedComponents(ee.Kernel.plus(1), 256)

stats_img = ee.Image.cat([
    blobs.select('labels'),
    d_ndvi, d_ndbi, d_mndwi,
    ee.Image.pixelArea().rename('area_m2'),
])

vectors = stats_img.reduceToVectors(
    geometry=AOI, scale=SCALE_M, geometryType='polygon',
    labelProperty='labels', reducer=ee.Reducer.mean().combine(ee.Reducer.sum(), '', True),
    maxPixels=1e10,
)

def add_props(f):
    area_ha = ee.Number(f.get('area_m2_sum')).divide(10000)
    return f.set({
        'area_ha': area_ha,
        'dNDVI': f.get('dNDVI_mean'), 'dNDBI': f.get('dNDBI_mean'), 'dMNDWI': f.get('dMNDWI_mean'),
        'mean_conf': ee.Number(1).min(area_ha.divide(50)),  # placeholder confidence until a real model provides one
    })

vectors = vectors.map(add_props).filter(ee.Filter.gte('area_ha', MIN_AREA_HA))
vectors = vectors.sort('area_ha', False)
n_found = vectors.size().getInfo()
top = ee.FeatureCollection(vectors.toList(MAX_REGIONS))
print(f'{n_found} regions found above {MIN_AREA_HA} ha; keeping the largest {min(n_found, MAX_REGIONS)}')
print(top.getInfo())

## Step 8 — Add location words and region ids (`r1`…`r5`), matching the web app's format

In [ ]:
import json

def where_in_aoi(lon, lat, aoi_bounds):
    w, s, e, n = aoi_bounds
    col = ['west', 'centre', 'east'][min(int(3 * (lon - w) / (e - w)), 2)]
    row = ['south', 'centre', 'north'][min(int(3 * (lat - s) / (n - s)), 2)]
    if row == 'centre' and col == 'centre': return 'centre'
    if row == 'centre': return col
    if col == 'centre': return row
    return f'{row}-{col}'

AOI_BOUNDS = [107.22, 20.98, 107.40, 21.10]
fc = top.getInfo()

regions = []
for i, feat in enumerate(fc['features'], 1):
    p = feat['properties']
    centroid = ee.Geometry(feat['geometry']).centroid(1).coordinates().getInfo()
    lon, lat = centroid
    regions.append({
        'id': f'r{i}',
        'area_ha': round(p['area_ha'], 1),
        'mean_conf': round(p['mean_conf'], 2),
        'dNDVI': round(p['dNDVI'], 3),
        'dNDBI': round(p['dNDBI'], 3),
        'dMNDWI': round(p['dMNDWI'], 3),
        'location': where_in_aoi(lon, lat, AOI_BOUNDS),
        'date_before': BEFORE_RANGE[0][:4], 'date_after': AFTER_RANGE[1][:4],
        'source': 'person1_threshold_v1',
        '_geometry': feat['geometry'],
    })

change_regions_geojson = {
    'type': 'FeatureCollection',
    'features': [{
        'type': 'Feature', 'id': i,
        'properties': {k: v for k, v in r.items() if k != '_geometry'},
        'geometry': r['_geometry'],
    } for i, r in enumerate(regions)],
}
print(json.dumps(change_regions_geojson, indent=2)[:1500])

## Step 9 — Save the GeoJSON (send this file to Person 3)
This has the exact same property names as `backend/demo_data.py` in the web app, so Person 3 only needs to swap the file — no code changes.

In [ ]:
out_path = f'{OUT}/change_regions_quangninh_v1.json'
with open(out_path, 'w') as f:
    json.dump(change_regions_geojson, f, indent=2)
print('Saved:', out_path)
print('Share this file (or the whole evichange Drive folder) with Person 3.')
print('It replaces CHANGE_REGIONS in backend/demo_data.py — property names already match.')

## Step 10 — Export the actual images too (optional but useful)
This saves the before/after pictures and the index-difference layers as GeoTIFFs to your Drive, in case Person 2 or Person 3 want the raw imagery (e.g. to show in the swipe tool, or to re-check something by eye).
Exports run in the background on Google's servers — check progress at **code.earthengine.google.com → Tasks**. Each can take a few minutes.

In [ ]:
def export(img, name, bands):
    task = ee.batch.Export.image.toDrive(
        image=img.select(bands), description=name, folder='evichange',
        fileNamePrefix=name, region=AOI, scale=SCALE_M, maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print('Started export:', name, '(check Tasks tab in the Earth Engine code editor for progress)')

export(before_img, 'quangninh_before_rgb', ['B4', 'B3', 'B2'])
export(after_img,  'quangninh_after_rgb',  ['B4', 'B3', 'B2'])
export(diff,        'quangninh_index_diff', ['dNDVI', 'dNDBI', 'dMNDWI'])

## What to do next
1. **Send `change_regions_quangninh_v1.json` to Person 3** (via the shared Drive folder, or attach it in your group chat).
2. **Check the map by eye.** Open `m4` above (Step 6) and see whether the red blobs look like real mining/construction change, not noise or cloud shadows. Adjust `MAX_CLOUD_PCT` or the date ranges if it looks wrong.
3. **This/next week: train a real model.** Use OSCD (same dataset Person 2 uses) to train U-Net or a similar change-detection model, then replace Step 6's threshold with your model's prediction. Steps 7–9 do not need to change.
4. **If your advisor provides an official Quang Ninh boundary or mining licence map**, swap it in at Step 3 and re-run — everything downstream updates automatically.